In [1]:
import subprocess
import os
import shutil
import time
os.environ['PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/bin:' + os.environ['PATH']
if 'LD_LIBRARY_PATH' in os.environ:
    os.environ['LD_LIBRARY_PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/lib:' + os.environ['LD_LIBRARY_PATH']
else:
    os.environ['LD_LIBRARY_PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/lib'
os.environ['GMX_MAXBACKUP'] = '-1'
os.environ['GMX_MAXCONSTRWARN'] = '-1'

import subprocess
import re

def run_mini(command, input_text=None):
    result = subprocess.run(command, capture_output=True, text=True, input=input_text)
    output = result.stdout + result.stderr
    print(output)
    for line in output.splitlines():
        if ("steepest descents converged to" in line.lower() or
            "fatal" in line.lower() or
            "error" in line.lower() or
            "steepest descents did not converge" in line.lower()):
            print(line)
            match = re.search(r'(\d+) steps', line)
            if match:
                steps = int(match.group(1))
                return steps == 5001
    return False

mini_prefix = "step4.0_minimization"
equi_prefix = "step4.0_minimization"
prod_prefix = "step5_production"
prod_step = "step5"

gro_directory = '/data/home/mrichte3/RNASeq/amide/'
gro_files = [f for f in os.listdir(gro_directory) if f.endswith('.gro')]
for input_gro in gro_files:
    start_time = time.time()
    command = ["gmx", "grompp", "-v", "-f", f"{mini_prefix}.mdp", "-o", f"{mini_prefix}.tpr", 
               "-c", f"{input_gro}", "-r", f"{input_gro}", 
               "-p", "topol.top", "-n", "index.ndx", "-maxwarn", "5"]
    run_mini(command)
    command = ["gmx", "mdrun", "-v", "-deffnm", mini_prefix, "-ntmpi", "1"]
    while not run_mini(command):
        pass
    new_file = "step4.0_minimization.gro"
    basename = os.path.splitext(os.path.basename(input_gro))[0]
    target_path = f"/data/home/mrichte3/RNASeq/amide/mini_output/{basename}.gro"
    if not os.path.exists(new_file):
        print(f"A problem occurred: {new_file} does not exist.#####################################################################")
    else:
        shutil.move(new_file, target_path)
##################

    elapsed_time = time.time() - start_time
    print(f"Process {basename} completed in {elapsed_time:.2f} seconds.")
    





                      :-) GROMACS - gmx grompp, 2024.2 (-:

Executable:   /data/home/mrichte3/gromacs-2024.2/install/bin/gmx
Data prefix:  /data/home/mrichte3/gromacs-2024.2/install
Working dir:  /data/home/mrichte3/RNASeq/test_alignment/md/working_folder
Command line:
  gmx grompp -v -f step4.0_minimization.mdp -o step4.0_minimization.tpr -c ENSG00000000003.gro -r ENSG00000000003.gro -p topol.top -n index.ndx -maxwarn 5


-------------------------------------------------------
Program:     gmx grompp, version 2024.2
Source file: src/gromacs/commandline/cmdlineparser.cpp (line 271)
Function:    void gmx::CommandLineParser::parse(int*, char**)

Error in user input:
Invalid command-line options
  In command-line option -c
    File 'ENSG00000000003.gro' does not exist or is not accessible.
    The file could not be opened.
      Reason: No such file or directory
      (call to fopen() returned error code 2)
  In command-line option -r
    File 'ENSG00000000003.gro' does not exist or is no

KeyboardInterrupt: 